In [1]:
import pandas as pd

In [2]:
file_path = r"C:\Users\Admin\Desktop\Workspace\Neo4j Project\Investor Awareness Pulse\Data Set\Unclean data\investor_awareness_pulse_raw_2500.csv"

In [3]:
df = pd.read_csv(file_path)

In [4]:
df.head()

,respondent_id,survey_date_raw,age_raw,gender_raw,education_raw,state_raw,area_type_raw,investment_info_source_raw,aware_of_event_raw,attended_event_raw,preferred_channel_raw,respondent_comment_raw
0,R0001,2026-06-14,26,male,NaN,Karnataka,Semi urban,College,I know the workshop,no,Whatsapp,"saw a poster, don't know how to register."
1,R0002,06/06/26,24,F,diploma course,maharashtra,Metro area,youtube,not yet,Not yet,WA,What is this workshop? Please share more details.
2,R0003,2026-09-27,17,Female,Postgraduate,jharkhand,metro,YouTube,not yet,No idea,YouTube,I am interested but was not aware before.
3,R0004,01/06/26,22,male,Bachelor's,Odisha,Semi-rural,financial institution,heard of it,participated,WhatsApp msg,would like to learn about mutual funds also.
4,R0005,09-Jun-2026,40,Man,Vocational,MH,Town,Friends,I know the workshop,Not yet,local bank,NaN


In [5]:
missing_count = df['respondent_id'].isna().sum()
print(missing_count)

5


In [6]:
df['respondent_id'] = [f"R{i:04d}" for i in range(1, len(df) + 1)]

In [7]:
df.head()

,respondent_id,survey_date_raw,age_raw,gender_raw,education_raw,state_raw,area_type_raw,investment_info_source_raw,aware_of_event_raw,attended_event_raw,preferred_channel_raw,respondent_comment_raw
0,R0001,2026-06-14,26,male,NaN,Karnataka,Semi urban,College,I know the workshop,no,Whatsapp,"saw a poster, don't know how to register."
1,R0002,06/06/26,24,F,diploma course,maharashtra,Metro area,youtube,not yet,Not yet,WA,What is this workshop? Please share more details.
2,R0003,2026-09-27,17,Female,Postgraduate,jharkhand,metro,YouTube,not yet,No idea,YouTube,I am interested but was not aware before.
3,R0004,01/06/26,22,male,Bachelor's,Odisha,Semi-rural,financial institution,heard of it,participated,WhatsApp msg,would like to learn about mutual funds also.
4,R0005,09-Jun-2026,40,Man,Vocational,MH,Town,Friends,I know the workshop,Not yet,local bank,NaN


In [8]:
df['survey_date_raw'] = pd.to_datetime(df['survey_date_raw'], format='mixed', errors='coerce')

In [9]:
df['survey_date_raw'] = df['survey_date_raw'].fillna(df['survey_date_raw'].mode()[0])

In [3]:
min_date = df['survey_date_raw'].min()
print("Minimum Date:", min_date)
max_date = df['survey_date_raw'].max()
print("Maximum Date:", max_date)

NameError: name 'df' is not defined

In [10]:
invalid_age = df[pd.to_numeric(df['age_raw'], errors='coerce').isna() & df['age_raw'].notna()]
invalid_age[['respondent_id', 'age_raw']]

,respondent_id,age_raw
8,R0009,26 yrs
9,R0010,about 52
10,R0011,about 19
12,R0013,about 48
13,R0014,29+
...,...,...
2476,R2477,32 yrs
2481,R2482,about 57
2487,R2488,25 yrs
2495,R2496,38 yrs


In [11]:
df['age_raw'] = df['age_raw'].astype(str).str.extract(r'(\d+)').astype(float)
df['age_raw'] = df['age_raw'].fillna(df['age_raw'].mean().round())
df['age_raw'] = df['age_raw'].astype(int)

In [12]:
df['gender_raw'].value_counts(dropna=False)

gender_raw
M                    282
Woman                267
F                    251
female               245
Female               242
Male                 237
male                 231
Man                  225
femle                225
male                 217
NaN                   27
Non-binary             9
Prefer not to say      9
unknown                8
non binary             8
No answer              7
Rather not say         5
NB                     5
Name: count, dtype: int64

In [13]:
df['gender_raw'] = df['gender_raw'].astype(str).str.strip()
gender_map = {
    'M': 'Male', 'Male': 'Male', 'male': 'Male', 'Man': 'Male',
    'F': 'Female', 'Female': 'Female', 'female': 'Female', 'Woman': 'Female', 'femle': 'Female',
    'Non-binary': 'Non-binary', 'non binary': 'Non-binary', 'NB': 'Non-binary',
    'nan': 'Undisclosed', 'unknown': 'Undisclosed', 'Prefer not to say': 'Undisclosed', 
    'No answer': 'Undisclosed', 'Rather not say': 'Undisclosed', 'Undisclosed': 'Undisclosed'
    }
df['gender_raw'] = df['gender_raw'].map(gender_map).fillna('Undisclosed')

In [14]:
df['gender_raw'].value_counts()


gender_raw
Female         1230
Male           1192
Undisclosed      56
Non-binary       22
Name: count, dtype: int64

In [15]:
df['gender_raw'].value_counts(dropna=False)

gender_raw
Female         1230
Male           1192
Undisclosed      56
Non-binary       22
Name: count, dtype: int64

In [16]:
df['education_raw'].value_counts(dropna=False)

education_raw
Graduate            204
Undergraduate       201
bachelors degree    197
UG                  182
Bachelor's          158
Post grad           124
10th/12th           124
School              119
Master's degree     116
Masters             114
PG                  112
secondary school    112
Secondary           108
High school         107
Postgraduate        105
Vocational          101
diploma course       94
Diploma              91
ITI                  82
NaN                  26
not mentioned        12
other                11
Name: count, dtype: int64

In [17]:
df['education_raw'] = df['education_raw'].astype(str).str.strip()
edu_map = {
    # School Category
    '10th/12th': 'School', 'School': 'School', 'secondary school': 'School', 
    'Secondary': 'School', 'High school': 'School',
    
    # Undergraduate Category
    'Undergraduate': 'Undergraduate', 'bachelors degree': 'Undergraduate', 
    'UG': 'Undergraduate', "Bachelor's": 'Undergraduate',
    
    # Postgraduate Category
    'Post grad': 'Postgraduate', "Master's degree": 'Postgraduate', 
    'Masters': 'Postgraduate', 'PG': 'Postgraduate', 'Postgraduate': 'Postgraduate',
    
    # Diploma / Vocational Category
    'Vocational': 'Diploma/Vocational', 'diploma course': 'Diploma/Vocational', 
    'Diploma': 'Diploma/Vocational', 'ITI': 'Diploma/Vocational',
    
    # Other & Missing
    'other': 'Other',
    'nan': 'Undisclosed', 'not mentioned': 'Undisclosed'
}

df['education_raw'] = df['education_raw'].map(edu_map).fillna('Undisclosed')


In [18]:
df['education_raw'].value_counts()

education_raw
Undergraduate         738
Postgraduate          571
School                570
Diploma/Vocational    368
Undisclosed           242
Other                  11
Name: count, dtype: int64

In [19]:
df['state_raw'].value_counts(dropna=False)

state_raw
Bihar             109
bihar             109
U.P.              104
UP                 95
Uttar Pradesh      95
Uttar pradesh      92
uttar pradesh      92
maharashtra        74
Gujarat            74
rajasthan          70
MH                 65
gujarat            60
Rajasthan          59
West Bengal        57
Karnataka          56
NCT Delhi          54
delhi              54
Telangana          52
New Delhi          51
WB                 50
Maharashtra        50
karnataka          49
Delhi              49
KA                 46
telangana          44
kerala             44
Tamil Nadu         44
haryana            43
west bengal        42
Madhya Pradesh     40
TN                 40
tamil nadu         39
Haryana            37
M.P.               36
Orissa             31
Punjab             31
Assam              30
MP                 28
Kerala             28
assam              28
Uttaranchal        27
Odisha             26
punjab             26
Jharkhand          26
jharkhand          25


In [20]:
df['state_raw'] = df['state_raw'].astype(str).str.strip().str.title()
state_fixes = {
    'U.P.': 'Uttar Pradesh', 'Up': 'Uttar Pradesh', 'Up/Delhi': 'Uttar Pradesh',
    'Mh': 'Maharashtra', 'M.P.': 'Madhya Pradesh', 'Mp': 'Madhya Pradesh',
    'Wb': 'West Bengal', 'Ka': 'Karnataka',
    'Nct Delhi': 'Delhi', 'New Delhi': 'Delhi',
    'Tn': 'Tamil Nadu', 'Orissa': 'Odisha', 'Chattisgarh': 'Chhattisgarh', 'Uttaranchal': 'Uttarakhand',
    'India': 'Undisclosed', 'Unknown': 'Undisclosed', 'Nan': 'Undisclosed'
}
df['state_raw'] = df['state_raw'].replace(state_fixes).fillna('Undisclosed')

In [21]:
df['state_raw'].value_counts()

state_raw
Uttar Pradesh     481
Bihar             218
Delhi             208
Maharashtra       189
Karnataka         151
West Bengal       149
Gujarat           134
Rajasthan         129
Tamil Nadu        123
Madhya Pradesh    104
Telangana          96
Odisha             81
Haryana            80
Kerala             72
Uttarakhand        60
Assam              58
Punjab             57
Jharkhand          51
Chhattisgarh       33
Undisclosed        26
Name: count, dtype: int64

In [22]:
df['area_type_raw'].value_counts(dropna=False)

area_type_raw
Urban           194
Urban area      190
rural           189
Town            179
Rural           174
Rural area      172
City            164
Semi-rural      163
urban           161
Village         159
Semi urban      153
Metro           131
Metro area      120
Metropolitan    119
metro           108
Metro city      101
NaN              12
not sure          7
village/town      4
Name: count, dtype: int64

In [23]:
df['area_type_raw'] = df['area_type_raw'].astype(str).str.strip().str.title()
area_fixes = {
    # Urban
    'Urban Area': 'Urban', 'City': 'Urban', 'Metro': 'Urban', 
    'Metro Area': 'Urban', 'Metropolitan': 'Urban', 'Metro City': 'Urban',
    
    # Semi-Urban
    'Semi Urban': 'Semi-Urban', 'Semi-Urban': 'Semi-Urban', 'Town': 'Semi-Urban', 'Village/Town': 'Semi-Urban',
    'Not Sure': 'Semi-Urban', 'Nan': 'Semi-Urban',
    
    # Rural
    'Rural Area': 'Rural', 'Village': 'Rural', 'Semi-Rural': 'Rural'
}
df['area_type_raw'] = df['area_type_raw'].replace(area_fixes).fillna('Semi-Urban')


In [24]:
df['area_type_raw'].value_counts()

area_type_raw
Urban         1288
Rural          857
Semi-Urban     355
Name: count, dtype: int64

In [26]:
df['investment_info_source_raw'].value_counts(dropna=False)

investment_info_source_raw
YouTube                             138
You Tube                            128
youtube                             122
youtube video                       113
Instagram                            93
bank poster                          92
WhatsApp forward                     91
friend                               90
relative                             86
Bank                                 84
Friends                              79
social-media                         79
Family                               78
Bank branch                          75
Facebook                             75
Social media                         74
financial institution                71
family/friends                       64
college notice                       56
TV/news                              53
official investor education page     53
paper                                49
workplace                            49
news                                 48
consultant   

In [27]:
df['investment_info_source_raw'] = df['investment_info_source_raw'].astype(str).str.strip()
source_map = {
    # 1. Digital & Social Media
    'YouTube': 'Digital & Social Media', 'You Tube': 'Digital & Social Media', 
    'youtube': 'Digital & Social Media', 'youtube video': 'Digital & Social Media',
    'Instagram': 'Digital & Social Media', 'WhatsApp forward': 'Digital & Social Media',
    'social-media': 'Digital & Social Media', 'Facebook': 'Digital & Social Media', 
    'Social media': 'Digital & Social Media',
    
    # 2. Friends & Family 
    'friend': 'Friends & Family', 'Friends': 'Friends & Family',
    'relative': 'Friends & Family', 'Family': 'Friends & Family', 
    'family/friends': 'Friends & Family', 'other source': 'Friends & Family', 
    'Other': 'Friends & Family', "don't remember": 'Friends & Family',
    'nan': 'Friends & Family', 'unknown': 'Friends & Family', 'not remember': 'Friends & Family',
    
    # 3. Banks & Financial Institutions
    'bank poster': 'Banks & Financial Institutions', 'Bank': 'Banks & Financial Institutions',
    'Bank branch': 'Banks & Financial Institutions', 'financial institution': 'Banks & Financial Institutions',
    'consultant': 'Banks & Financial Institutions', 'financial adviser': 'Banks & Financial Institutions',
    'Financial advisor': 'Banks & Financial Institutions',
    
    # 4. Traditional Media & News
    'TV/news': 'Traditional Media & News', 'paper': 'Traditional Media & News',
    'news': 'Traditional Media & News', 'TV': 'Traditional Media & News',
    'Newspaper': 'Traditional Media & News',
    
    # 5. Official & Institutional
    'college notice': 'Official & Institutional', 'workplace': 'Official & Institutional',
    'government finance website': 'Official & Institutional', 'official finance website': 'Official & Institutional',
    'official website': 'Official & Institutional', 'College': 'Official & Institutional',
    'office': 'Official & Institutional', 'official investor education page': 'Official & Institutional'
}

df['investment_info_source_raw'] = df['investment_info_source_raw'].map(source_map).fillna('Friends & Family')

In [30]:
short_names = {
    'Digital & Social Media': 'Digital',
    'Friends & Family': 'Personal',
    'Banks & Financial Institutions': 'Financial',
    'Official & Institutional': 'Official',
    'Traditional Media & News': 'Media'
}
df['investment_info_source_raw'] = df['investment_info_source_raw'].replace(short_names)


In [31]:
df['investment_info_source_raw'].value_counts()

investment_info_source_raw
Digital      913
Personal     537
Financial    437
Official     373
Media        240
Name: count, dtype: int64

In [32]:
df['aware_of_event_raw'].value_counts(dropna=False)

aware_of_event_raw
I know the workshop    196
Heard about it         185
Yes                    174
heard of it            173
Aware                  170
yes                    158
N                      152
not yet                150
yes - saw poster       149
Y                      147
No idea                146
Not aware              141
Never heard            137
No                     126
Not sure               124
no                     110
NaN                     62
Name: count, dtype: int64

In [33]:
df['aware_of_event_raw'] = df['aware_of_event_raw'].astype(str).str.strip()

aware_map = {

    'I know the workshop': 'Yes', 'Heard about it': 'Yes', 'Yes': 'Yes', 
    'heard of it': 'Yes', 'Aware': 'Yes', 'yes': 'Yes', 'yes - saw poster': 'Yes', 'Y': 'Yes',
    
   
    'N': 'No', 'not yet': 'No', 'No idea': 'No', 'Not aware': 'No', 
    'Never heard': 'No', 'No': 'No', 'Not sure': 'No', 'no': 'No', 
    'nan': 'No'
}
df['aware_of_event_raw'] = df['aware_of_event_raw'].map(aware_map).fillna('No')


In [34]:
df['aware_of_event_raw'].value_counts()

aware_of_event_raw
Yes    1352
No     1148
Name: count, dtype: int64

In [36]:
df['attended_event_raw'].value_counts(dropna=False)

attended_event_raw
not attended       335
Not yet            314
N                  313
No idea            301
No, never          299
No                 296
no                 278
NaN                 49
Y                   47
yes, last month     42
attended            42
Yes                 38
Attended once       38
participated        37
maybe               27
not sure            25
attended once       19
Name: count, dtype: int64

In [37]:
df['attended_event_raw'] = df['attended_event_raw'].astype(str).str.strip()
attended_map = {
    'Y': 'Yes', 'yes, last month': 'Yes', 'attended': 'Yes', 'Yes': 'Yes', 
    'Attended once': 'Yes', 'participated': 'Yes', 'attended once': 'Yes',
    
    
    'not attended': 'No', 'Not yet': 'No', 'N': 'No', 'No idea': 'No', 
    'No, never': 'No', 'No': 'No', 'no': 'No', 'nan': 'No', 
    'maybe': 'No', 'not sure': 'No'
}

df['attended_event_raw'] = df['attended_event_raw'].map(attended_map).fillna('No')

In [38]:
df['attended_event_raw'].value_counts()

attended_event_raw
No     2237
Yes     263
Name: count, dtype: int64

In [39]:
df['preferred_channel_raw'].value_counts(dropna=False)

preferred_channel_raw
You Tube              177
YouTube               164
youtube               149
Whatsapp              119
WA                    118
social-media          113
Facebook              111
WhatsApp              108
whatsapp              105
Instagram             104
Social media          103
WhatsApp msg           98
Workshop               93
offline session        93
e-mail                 92
In person              90
Email                  86
email                  82
sms                    75
in-person workshop     71
SMS                    69
text message           56
Bank branch            38
bank                   36
local bank             30
not sure               24
no preference          23
NaN                    21
Newspaper              19
newspaper              17
paper                  16
Name: count, dtype: int64

In [40]:
df['preferred_channel_raw'] = df['preferred_channel_raw'].astype(str).str.strip()
channel_map = {
    # 1. Social Media
    'You Tube': 'Social Media', 'YouTube': 'Social Media', 'youtube': 'Social Media',
    'Whatsapp': 'Social Media', 'WA': 'Social Media', 'WhatsApp': 'Social Media', 
    'whatsapp': 'Social Media', 'WhatsApp msg': 'Social Media',
    'social-media': 'Social Media', 'Facebook': 'Social Media', 
    'Instagram': 'Social Media', 'Social media': 'Social Media',
    
    # 2. Offline / In Person
    'Workshop': 'Offline', 'offline session': 'Offline', 'In person': 'Offline', 
    'in-person workshop': 'Offline', 'Bank branch': 'Offline', 'bank': 'Offline', 
    'local bank': 'Offline',
    
    # 3. Email & SMS
    'e-mail': 'Email & SMS', 'Email': 'Email & SMS', 'email': 'Email & SMS', 
    'sms': 'Email & SMS', 'SMS': 'Email & SMS', 'text message': 'Email & SMS',
    'Newspaper': 'Email & SMS', 'newspaper': 'Email & SMS', 'paper': 'Email & SMS',
    
    # 4. Personal
    'not sure': 'Personal', 'no preference': 'Personal', 'nan': 'Personal'
}

df['preferred_channel_raw'] = df['preferred_channel_raw'].map(channel_map).fillna('Personal')



In [41]:
df['preferred_channel_raw'].value_counts()

preferred_channel_raw
Social Media    1469
Email & SMS      512
Offline          451
Personal          68
Name: count, dtype: int64

In [42]:
df['respondent_comment_raw'].value_counts(dropna=False)

respondent_comment_raw
NaN                                                     471
I am interested but was not aware before.               142
If it is on Sunday I may attend.                        137
The message reached me too late.                        136
Heard about it from a friend but missed the date.       135
Please send details through WhatsApp.                   135
Saw a poster, don't know how to register.               134
Didn't see any information about it.                    133
I would like to know if it is free.                     128
I knew about the workshop but not the venue.            127
What is this workshop? Please share more details.       126
Please send details in Hindi/local language.            126
Need information in simple language.                    121
Need more practical examples and time for questions.     28
Would like to learn about mutual funds also.             28
Good information, some terms were difficult.             26
I attended once; 

In [43]:
df['respondent_comment_raw'] = df['respondent_comment_raw'].astype(str).str.strip().str.capitalize()
df['respondent_comment_raw'] = df['respondent_comment_raw'].replace({'Nan': 'No comment'}).fillna('No comment')


In [44]:
df['respondent_comment_raw'].value_counts().head()

respondent_comment_raw
No comment                                   471
If it is on sunday i may attend.             157
I am interested but was not aware before.    155
Please send details through whatsapp.        154
Saw a poster, don't know how to register.    154
Name: count, dtype: int64

In [46]:
df.columns = df.columns.str.replace('_raw', '', case=False)

In [47]:
print(list(df.columns))

['respondent_id', 'survey_date', 'age', 'gender', 'education', 'state', 'area_type', 'investment_info_source', 'aware_of_event', 'attended_event', 'preferred_channel', 'respondent_comment']


In [48]:
output_path = r"C:\Users\Admin\Desktop\Workspace\Neo4j Project\Investor Awareness Pulse\Data Set\Unclean data\investor_awareness_pulse_CLEANED.csv"
df.to_csv(output_path, index=False)


In [49]:
print(output_path)

C:\Users\Admin\Desktop\Workspace\Neo4j Project\Investor Awareness Pulse\Data Set\Unclean data\investor_awareness_pulse_CLEANED.csv


In [50]:
df.head

<bound method NDFrame.head of      respondent_id survey_date  age  gender           education        state  \
0            R0001  2026-06-14   26    Male         Undisclosed    Karnataka   
1            R0002  2026-06-06   24  Female  Diploma/Vocational  Maharashtra   
2            R0003  2026-09-27   17  Female        Postgraduate    Jharkhand   
3            R0004  2026-01-06   22    Male       Undergraduate       Odisha   
4            R0005  2026-06-09   40    Male  Diploma/Vocational  Maharashtra   
...            ...         ...  ...     ...                 ...          ...   
2495         R2496  2026-08-08   38  Female        Postgraduate        Bihar   
2496         R2497  2026-09-22   23    Male         Undisclosed  Maharashtra   
2497         R2498  2026-07-14   29    Male         Undisclosed    Karnataka   
2498         R2499  2026-09-28   39    Male  Diploma/Vocational       Odisha   
2499         R2500  2026-08-15   86  Female  Diploma/Vocational      Gujarat   

       ar